# 04 Tuning: feature-group ablation (Day 8)

Question: how much does each engineered feature group change ROC-AUC and PR-AUC?

Method:
- Each group is tested two ways: **added alone to the base set** and **removed alone from the full set**.
- Identical stratified 5-fold CV every time, so differences are **paired by fold**.
- Two models (Logistic Regression, Random Forest), run **with and without `error_code`**.
- Only the training split is used. The test split is never touched here.
- A delta only counts as real when its mean is larger than the std of the per-fold differences.

In [9]:
import os
import sys

# Run from the repo root so `src` and `data/` resolve, whether the notebook
# is started from the repo root or from notebooks/.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print(os.getcwd())

d:\ml-engineer-internship


In [10]:
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
from sklearn.ensemble import RandomForestClassifier  # noqa: E402
from sklearn.linear_model import LogisticRegression  # noqa: E402
from sklearn.model_selection import StratifiedKFold, cross_validate  # noqa: E402

from src.features import HISTORY_FEATURES, RATIO_FEATURES, TIME_FEATURES  # noqa: E402
from src.pipeline import (  # noqa: E402
    CATEGORICAL_FEATURES,
    NUMERIC_FEATURES,
    SEED,
    build_pipeline,
    load_splits,
    split_xy,
)

## Data (training split only)

In [11]:
train, _val, _test = load_splits(with_features=True)
ENGINEERED = RATIO_FEATURES + TIME_FEATURES + HISTORY_FEATURES
x_train, y_train = split_xy(train, extra=ENGINEERED + ["account_id"])
print(x_train.shape, round(y_train.mean(), 4))

(17500, 21) 0.0391


## CV and models

These must match `03_baselines.ipynb` (same fold seed, same model settings) so results are comparable with Day 7. Edit this cell if Day 7 used different settings.

In [12]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
MODELS = {
    "logreg": lambda: LogisticRegression(max_iter=1000, class_weight="balanced"),
    "rf": lambda: RandomForestClassifier(
        n_estimators=100, class_weight="balanced", random_state=SEED, n_jobs=-1
    ),
}
SCORING = {"roc": "roc_auc", "pr": "average_precision"}

## Feature groups and pipeline config

In [13]:
GROUPS = {
    "ratio": {"num": RATIO_FEATURES},
    "time": {"num": TIME_FEATURES},
    "history": {"num": HISTORY_FEATURES},
    "freq": {"freq": ["account_id"]},
    "target": {"target": ["account_id"]},
    "bins": {"bin": ["latency_ms", "retry_count"]},
}


def pick(groups, key):
    """Collect the columns of one kind from the chosen groups (None if empty)."""
    cols = [c for g in groups for c in GROUPS[g].get(key, [])]
    return cols or None


def make_kwargs(groups, use_error_code):
    """Translate a set of feature groups into build_pipeline arguments."""
    cat = [c for c in CATEGORICAL_FEATURES if use_error_code or c != "error_code"]
    num = list(NUMERIC_FEATURES)
    for g in groups:
        num += GROUPS[g].get("num", [])
    return {
        "num_cols": num,
        "cat_cols": cat,
        "freq_cols": pick(groups, "freq"),
        "target_cols": pick(groups, "target"),
        "bin_cols": pick(groups, "bin"),
    }


def fold_scores(model_name, groups, use_error_code):
    """Per-fold ROC-AUC and PR-AUC on the same folds every time."""
    pipe = build_pipeline(MODELS[model_name](), **make_kwargs(groups, use_error_code))
    res = cross_validate(pipe, x_train, y_train, cv=cv, scoring=SCORING)
    return {"roc": res["test_roc"], "pr": res["test_pr"]}

## Run the ablation (takes a few minutes)

In [14]:
rows = []
all_groups = list(GROUPS)
for use_ec in (True, False):
    for model in MODELS:
        base = fold_scores(model, [], use_ec)
        full = fold_scores(model, all_groups, use_ec)
        cases = [("ALL", "add_to_base", full, base)]
        for g in all_groups:
            added = fold_scores(model, [g], use_ec)
            dropped = fold_scores(model, [x for x in all_groups if x != g], use_ec)
            cases.append((g, "add_to_base", added, base))
            cases.append((g, "drop_from_full", full, dropped))
        for group, mode, with_g, without_g in cases:
            for metric in ("roc", "pr"):
                diff = with_g[metric] - without_g[metric]  # paired by fold
                rows.append(
                    {
                        "error_code": use_ec,
                        "model": model,
                        "group": group,
                        "mode": mode,
                        "metric": metric,
                        "mean": diff.mean(),
                        "std": diff.std(ddof=1),
                    }
                )
results = pd.DataFrame(rows)
results["verdict"] = np.select(
    [results["mean"] > results["std"], results["mean"] < -results["std"]],
    ["helps", "hurts"],
    default="no clear effect",
)

KeyboardInterrupt: 

## Ablation table

Values are the mean change in the metric (positive means the group helps). `add_to_base` = group added alone to the base features. `drop_from_full` = full set minus that group.

In [ ]:
os.makedirs("reports", exist_ok=True)
results.round(4).to_csv("reports/ablation_results.csv", index=False)
table = results.pivot_table(
    index=["error_code", "model", "group"],
    columns=["mode", "metric"],
    values="mean",
).round(4)
table

mode                      add_to_base         drop_from_full        
metric                             pr     roc             pr     roc
error_code model  group                                             
False      logreg ALL          0.0034 -0.0015            NaN     NaN
                  bins         0.0041  0.0007         0.0050  0.0000
                  freq         0.0000 -0.0009        -0.0002 -0.0009
                  history     -0.0006 -0.0014        -0.0003 -0.0006
                  ratio       -0.0006 -0.0009        -0.0003 -0.0013
                  target       0.0009  0.0009         0.0009  0.0004
                  time        -0.0006  0.0006        -0.0004  0.0004
           rf     ALL          0.0103  0.0131            NaN     NaN
                  bins        -0.0055 -0.0010         0.0044  0.0024
                  freq         0.0004  0.0072         0.0002 -0.0034
                  history     -0.0001  0.0000        -0.0020 -0.0023
                  ratio        0.0045  0.0027        -0.0005  0.0039
                  target       0.0052  0.0141         0.0005 -0.0041
                  time         0.0114  0.0130        -0.0011  0.0084
True       logreg ALL         -0.0057 -0.0056            NaN     NaN
                  bins         0.0007 -0.0036         0.0002 -0.0028
                  freq        -0.0004 -0.0006        -0.0007 -0.0006
                  history     -0.0017 -0.0019        -0.0006 -0.0016
                  ratio       -0.0021 -0.0001        -0.0022 -0.0001
                  target       0.0007  0.0003         0.0004  0.0010
                  time        -0.0021 -0.0005        -0.0023 -0.0007
           rf     ALL          0.0095 -0.0070            NaN     NaN
                  bins        -0.0058 -0.0058        -0.0094 -0.0037
                  freq         0.0151  0.0050        -0.0045 -0.0004
                  history      0.0099 -0.0088        -0.0038 -0.0076
                  ratio        0.0054  0.0076        -0.0029 -0.0013
                  target       0.0107  0.0053         0.0003 -0.0023
                  time         0.0061 -0.0084        -0.0122 -0.0048

## PR-AUC verdicts

`helps` / `hurts` only when the mean delta is larger than the std of the per-fold differences. Otherwise `no clear effect`.

In [ ]:
results[results["metric"] == "pr"].round(4).sort_values(
    ["error_code", "model", "mode", "mean"], ascending=[True, True, True, False]
)

,error_code,model,group,mode,metric,mean,std,verdict
75,False,logreg,bins,add_to_base,pr,0.0041,0.0040,helps
53,False,logreg,ALL,add_to_base,pr,0.0034,0.0079,no clear effect
71,False,logreg,target,add_to_base,pr,0.0009,0.0006,helps
67,False,logreg,freq,add_to_base,pr,0.0000,0.0004,no clear effect
55,False,logreg,ratio,add_to_base,pr,-0.0006,0.0042,no clear effect
59,False,logreg,time,add_to_base,pr,-0.0006,0.0017,no clear effect
63,False,logreg,history,add_to_base,pr,-0.0006,0.0006,hurts
77,False,logreg,bins,drop_from_full,pr,0.0050,0.0034,helps
73,False,logreg,target,drop_from_full,pr,0.0009,0.0005,helps
69,False,logreg,freq,drop_from_full,pr,-0.0002,0.0002,no clear effect


## Conclusion

The engineered features did not give a clear, repeatable lift. All PR-AUC changes are at most about 1.5 points and most are smaller than the fold-to-fold std, so I do not claim a gain.

- **All groups together** vs base: PR-AUC +0.0034 (logreg) and +0.0103 (RF) without error_code; -0.0057 and +0.0095 with it. Every value is within one std (0.008-0.015): no clear effect.
- **Bins** is the only group with a consistent sign in both tests: logreg without error_code, +0.0041 (add) and +0.0050 (drop), std 0.003-0.004. The gain is small and disappears when error_code is present, so it is a weak finding. For RF, bins is negative or inconsistent.
- **Freq and target encoding** show isolated "helps" labels (RF freq +0.0151 with error_code, RF target +0.0052 without), but the matching drop test disagrees or is within noise. I treat these as noise.
- **History** features, which I expected to help most, show no gain (slightly negative for logreg, about zero for RF).
- **error_code** dominates: with it present, group effects are even less clear.

Caveats: 5 folds and about 3.9% positives make per-fold PR-AUC noisy. The "mean > std" rule is lenient, and with 52 PR-AUC comparisons several flags are expected by chance, so only results that agree between add-to-base and drop-from-full are treated as evidence. The ablation covers logreg and RF only; tree boosting (Day 9) may use these features differently.

Decision: keep the feature code and tests (the pipeline is leak-free and ready), but do not present the features as a proven improvement. Day 9 will choose the final feature set by CV on the boosted models.